# FedCardioTwin — Kaggle (T4 × 2)

**Dataset:** attach `hammad0110/ieee-2` as input.
**GPU:** select *2 × T4* in Settings → Accelerator.

| Phase | GPUs | What runs | ~Wall clock |
|---|---|---|---|
| 1 — parallel | GPU 0 + GPU 1 | centralized \| twin | 2 h |
| 2 — parallel | GPU 0 + GPU 1 | federated ×7 strategies (4 \| 3) | 6–8 h |
| Session 1 end — **Save & Run All** to commit checkpoints | | | |
| 3 — parallel | GPU 0 + GPU 1 | loho \| conformal | 4–5 h |
| 4 — sequential | GPU 0 | ensemble + tables | 0.5 h |

Re-attach the committed output as `ieee-2-run` in the next session to resume.

In [ ]:
# 1. Clone repo + install
import os, sys, subprocess

REPO_URL = 'https://github.com/perfectenshclag/FedCardioTwin-.git'
REPO_ROOT = '/kaggle/working/FedCardioTwin'

if not os.path.exists(f'{REPO_ROOT}/fedcardiotwin'):
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_ROOT], check=True)

os.chdir(REPO_ROOT)
sys.path.insert(0, REPO_ROOT)
!pip install -q -r requirements.txt

if not os.path.exists('external/evaluation-2021'):
    !git clone -q --depth 1 https://github.com/physionetchallenges/evaluation-2021 external/evaluation-2021

import torch
print('CUDA:', torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}:', torch.cuda.get_device_name(i))

In [ ]:
# 2. Paths + parallel runner
import os, sys, shutil, subprocess, threading

# Find cache wherever Kaggle mounted it (path varies by how dataset was added)
_hits = __import__('glob').glob('/kaggle/input/**/CPSC/X.npy', recursive=True)
if not _hits:
    raise RuntimeError('Cache not found — make sure hammad0110/ieee-2 is attached.')
CACHE_SRC = __import__('os').path.dirname(__import__('os').path.dirname(_hits[0]))
print('Cache found at:', CACHE_SRC)
CKPT_DIR    = '/kaggle/working/checkpoints'
RESULTS_DIR = '/kaggle/working/results'

for d in [CKPT_DIR, RESULTS_DIR]: os.makedirs(d, exist_ok=True)

# Symlink cache (read-only dataset)
os.makedirs('data', exist_ok=True)
if not os.path.islink('data/cache') and not os.path.exists('data/cache'):
    os.symlink(CACHE_SRC, 'data/cache')

# Symlink output dirs so scripts can use relative paths
for name, target in [('checkpoints', CKPT_DIR), ('results', RESULTS_DIR)]:
    if not os.path.islink(name) and not os.path.exists(name):
        os.symlink(target, name)

# Resume from a previously committed run (re-attach output as 'ieee-2-run')
PREV = '/kaggle/input/ieee-2-run/checkpoints'
if os.path.isdir(PREV):
    copied = sum(1 for f in os.listdir(PREV)
                 if not os.path.exists(f'{CKPT_DIR}/{f}')
                 and shutil.copy2(f'{PREV}/{f}', f'{CKPT_DIR}/{f}') is None)
    print(f'Resumed {copied} checkpoint(s) from previous run.')

print('Cache    :', CACHE_SRC)
print('Ckpts    :', CKPT_DIR)
print('Results  :', RESULTS_DIR)

# ── parallel runner ───────────────────────────────────────────────────────────
def _stream(pipe, tag):
    for raw in iter(pipe.readline, b''):
        print(f'[{tag}] {raw.decode(errors="replace").rstrip()}', flush=True)

def run_parallel(jobs):
    """jobs: list of (gpu_id, tag, extra_args_for_run_experiments.py)"""
    procs, threads = [], []
    C = ['--ckpt-dir', CKPT_DIR, '--results-dir', RESULTS_DIR]
    for gpu_id, tag, extra in jobs:
        env = {**os.environ, 'CUDA_VISIBLE_DEVICES': str(gpu_id)}
        p = subprocess.Popen([sys.executable, 'scripts/run_experiments.py'] + extra + C,
                             env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        t = threading.Thread(target=_stream, args=(p.stdout, tag), daemon=True)
        t.start()
        procs.append(p); threads.append(t)
    for p in procs: p.wait()
    for t in threads: t.join(timeout=3)
    codes = {j[1]: p.returncode for j, p in zip(jobs, procs)}
    for tag, rc in codes.items():
        print(f'[{tag}] exit {rc}' + (' ✓' if rc == 0 else ' ✗ — check output above'))
    return codes

In [ ]:
# 3. Smoke test (~2 min)
!python tests/smoke_test.py

In [ ]:
# 4. Cache verification
import numpy as np
for c in ['CPSC','CPSC-Extra','Georgia','Chapman','Ningbo','PTB-XL','PTBXL_TRACKB']:
    X = np.load(f'data/cache/{c}/X.npy', mmap_mode='r')
    Y = np.load(f'data/cache/{c}/Y.npy', mmap_mode='r')
    print(f'{c:16s} X={str(X.shape):20s} {X.dtype}   Y={str(Y.shape):16s} {Y.dtype}')

In [ ]:
# 5. FAST sanity (~15 min on one GPU)
C = f'--ckpt-dir {CKPT_DIR} --results-dir {RESULTS_DIR}'
!CUDA_VISIBLE_DEVICES=0 python scripts/run_experiments.py --stage centralized --preset fast {C}
!CUDA_VISIBLE_DEVICES=0 python scripts/run_experiments.py --stage federated --preset fast --strategies fedavg fedbn {C}

In [ ]:
# 6a. Phase 1 — centralized (GPU 0) | twin (GPU 1)  [~2 h]
# centralized and twin are independent; run simultaneously.
run_parallel([
    (0, 'GPU0/centralized', ['--stage', 'centralized', '--preset', 'full']),
    (1, 'GPU1/twin',        ['--stage', 'twin',        '--preset', 'full']),
])

In [ ]:
# 6b. Phase 2 — federated ×7 strategies split across both GPUs  [~6–8 h]
# GPU 0: local · fedavg · fedprox · fedbn
# GPU 1: fedavgm · ditto · fedper
# Any seed/strategy already completed is skipped automatically.
run_parallel([
    (0, 'GPU0/fed', ['--stage', 'federated', '--preset', 'full',
                     '--strategies', 'local', 'fedavg', 'fedprox', 'fedbn']),
    (1, 'GPU1/fed', ['--stage', 'federated', '--preset', 'full',
                     '--strategies', 'fedavgm', 'ditto', 'fedper']),
])

---
### Session boundary (if needed)

If the 12 h Kaggle limit is near: **Save & Run All** → this commits
`/kaggle/working/checkpoints/` and `/kaggle/working/results/` as output.
Next session: re-attach that output as `ieee-2-run` and re-run from cell 2
(paths + resume) to pick up where you left off.
---

In [ ]:
# 6c. Phase 3 — loho (GPU 0) | conformal (GPU 1)  [~4–5 h]
# conformal evaluates deployed per-client models from phase 2 (fedbn/fedavg seed 0).
# loho trains fresh leave-one-hospital-out models, independent of phase 2.
run_parallel([
    (0, 'GPU0/loho',      ['--stage', 'loho',      '--preset', 'full']),
    (1, 'GPU1/conformal', ['--stage', 'conformal']),
])

In [ ]:
# 6d. Seed ensemble — headline number
!python scripts/ensemble_eval.py \
    --ckpts {CKPT_DIR}/central_seed0.pt \
            {CKPT_DIR}/central_seed1.pt \
            {CKPT_DIR}/central_seed2.pt \
    --results-dir {RESULTS_DIR}

In [ ]:
# 6e. Paper tables (mean ± std + Wilcoxon vs FedAvg)
!python scripts/make_tables.py

## Output → paper mapping

| `results/` file | Paper artifact |
|---|---|
| `fed_*_seed*.json` | Table 2: per-client + MEAN/WORST AUROC/F1, comm MB/round; `history` → convergence figure |
| `centralized_seed*.json` | Table 2 top row: pooled upper bound |
| `ensemble.json` | Table 2 headline: 3-seed SE-Inception ensemble |
| `loho_seed*.json` | Generalization table: leave-one-hospital-out |
| `twin_seed*.json` | Table 3: cold vs warm AUROC, gain curve, alert AUROC, backward transfer |
| `twin_seed*_arrays.npz` | Raw per-record arrays for post-hoc analyses |
| `conformal.json` | Table 4: per-hospital FNR coverage + mean set size |
| `make_tables.py` stdout | Mean ± std + Wilcoxon significance vs FedAvg |

**Ablations:** toggle `se=False`, `augment=False`, `mixup_alpha=0`, `ema_decay=0` in `configs.py`,
or pass `--strategies fedavg` vs `fedbn`, or set `update_steps=0`/`replay_size=1` for twin.